In [41]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

In [42]:
#%cd ../..
#!ls

In [43]:
tmp_data = pd.read_csv('EDA/data/findata.csv', index_col=0)

In [44]:
tmp_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   float64
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   float6

In [45]:
go_data = tmp_data.copy()

In [46]:
# Удалим не нужные тут таргеты 
go_data = go_data.drop(columns=['IC50, mM', 'SI'])

In [47]:
# Выбираем самые полезные параметры 

# Рассчитываем корреляцию всех признаков 
go_correlations = go_data.corr()['CC50, mM'].abs().sort_values()

# Отбираем признаки с корреляцией больше 0.1 
gl_high_info_features = go_correlations[go_correlations > 0.1]

print("Информативные признаки (есть связь):")
gl_high_info_features = gl_high_info_features.drop(['CC50, mM'], errors='ignore')


#Для финальной модели оставляем только информативные параметры  
gl_final_param = gl_high_info_features.index.unique().tolist() 

print(len(gl_final_param))
display(go_correlations.sort_values(ascending=False).head(50))

display(gl_final_param)

Информативные признаки (есть связь):
56


CC50, mM               1.000000
MolMR                  0.310111
LabuteASA              0.309191
MolWt                  0.306439
ExactMolWt             0.306382
HeavyAtomCount         0.305169
Chi0                   0.304792
Chi1                   0.304380
HeavyAtomMolWt         0.303163
Kappa1                 0.302206
Chi1v                  0.301525
NumValenceElectrons    0.301473
Kappa2                 0.300471
Chi0v                  0.296823
Chi1n                  0.294735
Kappa3                 0.294513
FpDensityMorgan1       0.293989
Chi0n                  0.291414
Chi2v                  0.269558
BertzCT                0.262327
FpDensityMorgan2       0.256608
Chi2n                  0.254055
RingCount              0.251477
Chi4v                  0.251166
PEOE_VSA7              0.231089
Chi4n                  0.230276
MolLogP                0.225358
HallKierAlpha          0.215193
Chi3v                  0.210437
BCUT2D_CHGLO           0.205016
SMR_VSA10              0.204249
PEOE_VSA

['NumAliphaticRings',
 'qed',
 'SPS',
 'BCUT2D_MWLOW',
 'PEOE_VSA1',
 'AvgIpc',
 'SlogP_VSA5',
 'SMR_VSA5',
 'NOCount',
 'FpDensityMorgan3',
 'NumHeteroatoms',
 'MaxAbsPartialCharge',
 'FractionCSP3',
 'TPSA',
 'NumRotatableBonds',
 'NumHAcceptors',
 'MinPartialCharge',
 'BCUT2D_LOGPLOW',
 'EState_VSA4',
 'VSA_EState4',
 'SlogP_VSA6',
 'BalabanJ',
 'Chi3n',
 'VSA_EState2',
 'SMR_VSA1',
 'PEOE_VSA6',
 'SMR_VSA10',
 'BCUT2D_CHGLO',
 'Chi3v',
 'HallKierAlpha',
 'MolLogP',
 'Chi4n',
 'PEOE_VSA7',
 'Chi4v',
 'RingCount',
 'Chi2n',
 'FpDensityMorgan2',
 'BertzCT',
 'Chi2v',
 'Chi0n',
 'FpDensityMorgan1',
 'Kappa3',
 'Chi1n',
 'Chi0v',
 'Kappa2',
 'NumValenceElectrons',
 'Chi1v',
 'Kappa1',
 'HeavyAtomMolWt',
 'Chi1',
 'Chi0',
 'HeavyAtomCount',
 'ExactMolWt',
 'MolWt',
 'LabuteASA',
 'MolMR']

In [48]:
#перебором выявим параметры котрые коррелируют между собой > 60% и оставим только второй 

for col_1 in gl_final_param:
    for col_2 in gl_final_param:
        if col_1 != col_2:
            lv_correlation = go_data[col_1].corr(go_data[col_2])
            if lv_correlation >= 0.70:
                print(f' Параметр {col_1} коррелирует с парамтером {col_2} : {lv_correlation}')
                gl_final_param.remove(col_2)
display(gl_final_param)

 Параметр NumAliphaticRings коррелирует с парамтером Chi4n : 0.7104343085012803
 Параметр SPS коррелирует с парамтером FractionCSP3 : 0.8086104934413165
 Параметр SPS коррелирует с парамтером HallKierAlpha : 0.7260631084193936
 Параметр PEOE_VSA1 коррелирует с парамтером NOCount : 0.7845294067942807
 Параметр PEOE_VSA1 коррелирует с парамтером TPSA : 0.8410261554105481
 Параметр PEOE_VSA1 коррелирует с парамтером NumHAcceptors : 0.7867986550434035
 Параметр PEOE_VSA1 коррелирует с парамтером SMR_VSA1 : 0.8183629797769292
 Параметр AvgIpc коррелирует с парамтером BertzCT : 0.779775515975152
 Параметр AvgIpc коррелирует с парамтером Chi0v : 0.7216135922852
 Параметр AvgIpc коррелирует с парамтером NumValenceElectrons : 0.7130022298059346
 Параметр AvgIpc коррелирует с парамтером HeavyAtomMolWt : 0.7635405475123447
 Параметр AvgIpc коррелирует с парамтером Chi0 : 0.7268291740471673
 Параметр AvgIpc коррелирует с парамтером ExactMolWt : 0.7561044087314178
 Параметр AvgIpc коррелирует с пар

 Параметр FpDensityMorgan3 коррелирует с парамтером FpDensityMorgan2 : 0.9388232318246882
 Параметр FpDensityMorgan3 коррелирует с парамтером FpDensityMorgan1 : 0.7973776069503878
 Параметр NumHeteroatoms коррелирует с парамтером MolWt : 0.7073296556812635
 Параметр NumRotatableBonds коррелирует с парамтером Chi0n : 0.726903614110838
 Параметр NumRotatableBonds коррелирует с парамтером Chi1n : 0.7209587887362863
 Параметр NumRotatableBonds коррелирует с парамтером Kappa1 : 0.7443730432619066
 Параметр BCUT2D_LOGPLOW коррелирует с парамтером BCUT2D_CHGLO : 0.8673416403304163
 Параметр Chi4v коррелирует с парамтером Chi2v : 0.9264296360336696
 Параметр Chi4v коррелирует с парамтером Chi1v : 0.809035957567203
 Параметр RingCount коррелирует с парамтером Chi1 : 0.7208246313467956
 Параметр RingCount коррелирует с парамтером MolMR : 0.7006222488157218
 Параметр Kappa3 коррелирует с парамтером NumRotatableBonds : 0.7611703097339004
 Параметр Kappa3 коррелирует с парамтером Kappa2 : 0.9360263

['NumAliphaticRings',
 'qed',
 'SPS',
 'BCUT2D_MWLOW',
 'PEOE_VSA1',
 'AvgIpc',
 'SlogP_VSA5',
 'FpDensityMorgan3',
 'NumHeteroatoms',
 'MaxAbsPartialCharge',
 'MinPartialCharge',
 'BCUT2D_LOGPLOW',
 'EState_VSA4',
 'VSA_EState4',
 'SlogP_VSA6',
 'BalabanJ',
 'VSA_EState2',
 'PEOE_VSA6',
 'SMR_VSA10',
 'MolLogP',
 'PEOE_VSA7',
 'Chi4v',
 'RingCount',
 'Kappa3',
 'HeavyAtomCount']

In [49]:
go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CC50, mM             1001 non-null   float64
 1   MaxAbsEStateIndex    1001 non-null   float64
 2   MaxEStateIndex       1001 non-null   float64
 3   MinAbsEStateIndex    1001 non-null   float64
 4   MinEStateIndex       1001 non-null   float64
 5   qed                  1001 non-null   float64
 6   SPS                  1001 non-null   float64
 7   MolWt                1001 non-null   float64
 8   HeavyAtomMolWt       1001 non-null   float64
 9   ExactMolWt           1001 non-null   float64
 10  NumValenceElectrons  1001 non-null   float64
 11  MaxPartialCharge     1001 non-null   float64
 12  MinPartialCharge     1001 non-null   float64
 13  MaxAbsPartialCharge  1001 non-null   float64
 14  MinAbsPartialCharge  1001 non-null   float64
 15  FpDensityMorgan1     1001 non-null   float6

In [50]:
X = go_data[gl_final_param]
#X = go_data.drop(columns='IC50, mM')
y = go_data['CC50, mM']

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (700, 25), (700,)
Test dataset size: (301, 25), (301,)


In [52]:
import warnings
warnings.filterwarnings('ignore')

In [53]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

models = {
    "LinearRegression": (LinearRegression(), 
        {
            'fit_intercept': [True, False]
        }),

    "RidgeRegression": (Ridge(), 
        {
            'alpha': [0.1, 1.0, 10.0, 100.0],  
            'solver': ['auto', 'cholesky', 'sag', 'lsqr']
        }),

    "RandomForestRegressor": (RandomForestRegressor(), 
        {
            'n_estimators': (10, 200, 350),
            'max_depth': (3, 10, 25),
            'min_samples_split': (2, 10, 15, 50)

        })
    
}



In [54]:
from skopt import BayesSearchCV
from sklearn.metrics import silhouette_score
from sklearn.model_selection import PredefinedSplit

# Перебор моделей
best_global_score = -10
best_model = None
results_report = []


for name, (model, params) in models.items():
    print(f"Обучаем {name}...")

    # Байесовская оптимизация гиперпараметров
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=params,
        n_iter=35,
        cv=15,
        scoring='r2',  
        n_jobs=-1,
        random_state=42
    )

    # Обучение модели
    bayes_search.fit(X_train, y_train)

    # 
    score = bayes_search.best_score_  # type: ignore
    results_report.append({"Model": name, "Score": score, "Params": bayes_search.best_params_}) # type: ignore
    
    # Сохраняем абсолютного победителя
    if score > best_global_score:
        best_global_score = score
        best_model = bayes_search.best_estimator_ # type: ignore

# --- АНАЛИЗ ---
print("\n--- Report  ---")
print(pd.DataFrame(results_report))
print(f"\n Лучшая модель: {best_model}")



Обучаем LinearRegression...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

Обучаем RidgeRegression...
Обучаем RandomForestRegressor...

--- Report  ---
                   Model     Score  \
0       LinearRegression -0.057256   
1        RidgeRegression  0.152936   
2  RandomForestRegressor  0.510156   

                                              Params  
0                            {'fit_intercept': True}  
1                 {'alpha': 100.0, 'solver': 'lsqr'}  
2  {'max_depth': 10, 'min_samples_split': 2, 'n_e...  

 Лучшая модель: RandomForestRegressor(max_depth=10, n_estimators=200)


In [56]:
# на тестовой выборке 
from sklearn import metrics

y_pred = best_model.predict(X_test)  # type: ignore

print("MAE", metrics.mean_absolute_error(y_test, y_pred))
print("MSE", metrics.mean_squared_error(y_test, y_pred))
print("R2 Score:", best_model.score(X_test, y_test)) # type: ignore

MAE 0.4100707469118039
MSE 0.36926996660282724
R2 Score: 0.47018999095501124
